In [1]:
D1 = ['feel', 'good']
D2 = ['feel', 'better', 'thank']
D3 = ['happened', 'good']
vocabulary_vector = ['feel', 'good', 'better', 'thank', 'happened']

D_vec = []
for D in [D1, D2, D3]:
  # in raw python
  D_vec.append([1 if word in D else 0 for word in vocabulary_vector])

print(D_vec)
# D1_vector = [1 if word in D1 else 0 for word in vocabulary_vector]

[[1, 1, 0, 0, 0], [1, 0, 1, 1, 0], [0, 1, 0, 0, 1]]


In [2]:
import kagglehub # type: ignore
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")

print("Path to dataset files:", path)

# Read csv files from \News _dataset
fake_full = pd.read_csv(path + "\\News _dataset\\Fake.csv")
real_full = pd.read_csv(path + "\\News _dataset\\True.csv")

fake = fake_full.head(50)
real = real_full.head(50)

print("Fake news dataset shape:", fake.shape)
print("Real news dataset shape:", real.shape)

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\emineyetm\fake-news-detection-datasets\versions\1
Fake news dataset shape: (50, 4)
Real news dataset shape: (50, 4)


In [3]:
import string
import re

contractions = {
  "i'm": "i am",
  "you're": "you are",
  "he's": "he is",
  "she's": "she is",
  "it's": "it is",
  "we're": "we are",
  "they're": "they are",
  "i've": "i have",
  "you've": "you have",
  "we've": "we have",
  "they've": "they have",
  "i'd": "i would",
  "you'd": "you would",
  "he'd": "he would",
  "she'd": "she would",
  "we'd": "we would",
  "they'd": "they would",
  "i'll": "i will",
  "you'll": "you will",
  "he'll": "he will",
  "she'll": "she will",
  "we'll": "we will",
  "they'll": "they will",
  "isn't": "is not",
  "aren't": "are not",
  "wasn't": "was not",
  "weren't": "were not",
  "hasn't": "has not",
  "haven't": "have not",
  "hadn't": "had not",
  "won't": "will not",
  "wouldn't": "would not",
  "don't": "do not",
  "doesn't": "does not",
  "didn't": "did not",
  "can't": "cannot",
  "couldn't": "could not",
  "shouldn't": "should not",
  "that's": "that is",
  "there's": "there is",
  "what's": "what is",
  "let's": "let us",
  # "couldn t": "could not",
}

def expand_contraction(text):
  text = text.replace('\u2019', "'").replace('\u2018', "'")
  for cont, exp in contractions.items():
    text = re.sub(re.escape(cont), exp, text, flags=re.IGNORECASE)
  return text

# Convert into lowercase text
fake['text'] = fake['text'].str.lower()
real['text'] = real['text'].str.lower()

# Expand contraction from TP0
fake['text'] = fake['text'].apply(expand_contraction)
real['text'] = real['text'].apply(expand_contraction)

# Remove punctutations from 'text' column in dataframes

translator = str.maketrans({key: " " for key in string.punctuation})
fake['text'] = fake['text'].apply(lambda x: x.translate(translator))
real['text'] = real['text'].apply(lambda x: x.translate(translator))


# .apply(func) is for iteration accross all rows instead of classic loop

print(f'fake tokenized:\n{fake['text'].head(5)}')
print(f'\nreal tokenized:\n{real['text'].head(5)}')



fake tokenized:
0    donald trump just couldn t wish all americans ...
1    house intelligence committee chairman devin nu...
2    on friday  it was revealed that former milwauk...
3    on christmas day  donald trump announced that ...
4    pope francis used his annual christmas day mes...
Name: text, dtype: str

real tokenized:
0    washington  reuters    the head of a conservat...
1    washington  reuters    transgender people will...
2    washington  reuters    the special counsel inv...
3    washington  reuters    trump campaign adviser ...
4    seattle washington  reuters    president donal...
Name: text, dtype: str


In [ ]:
import spacy  # type: ignore

# loading minimal english model, and disable parser, ner since not needed
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def tokenize_text(text): return [token.text for token in nlp(text)]

# tokenization to 'text' column
fake['text_tokens'] = fake['text'].apply(tokenize_text)
real['text_tokens'] = real['text'].apply(tokenize_text)

print(f'fake tokenized:\n{fake['text_tokens'].head(5)}')
print(f'\nreal tokenized:\n{real['text_tokens'].head(5)}')




fake tokenized:
0    [donald, trump, just, couldn, t, wish, all, am...
1    [house, intelligence, committee, chairman, dev...
2    [on, friday,  , it, was, revealed, that, forme...
3    [on, christmas, day,  , donald, trump, announc...
4    [pope, francis, used, his, annual, christmas, ...
Name: text_tokens, dtype: object

real tokenized:
0    [washington,  , reuters,    , the, head, of, a...
1    [washington,  , reuters,    , transgender, peo...
2    [washington,  , reuters,    , the, special, co...
3    [washington,  , reuters,    , trump, campaign,...
4    [seattle, washington,  , reuters,    , preside...
Name: text_tokens, dtype: object


In [ ]:
# Text Tokenization
from nltk.tokenize import word_tokenize # type: ignore

def tokenize_text(text): return word_tokenize(text)

fake['cleaned_tokens'] = fake['text'].apply(tokenize_text)
real['cleaned_tokens'] = real['text'].apply(tokenize_text)

print(f'fake tokenized:\n{fake['cleaned_tokens'].head(5)}')
print(f'\nreal tokenized:\n{real['cleaned_tokens'].head(5)}')



fake tokenized:
0    [donald, trump, just, couldn, t, wish, all, am...
1    [house, intelligence, committee, chairman, dev...
2    [on, friday, it, was, revealed, that, former, ...
3    [on, christmas, day, donald, trump, announced,...
4    [pope, francis, used, his, annual, christmas, ...
Name: cleaned_tokens, dtype: object

real tokenized:
0    [washington, reuters, the, head, of, a, conser...
1    [washington, reuters, transgender, people, wil...
2    [washington, reuters, the, special, counsel, i...
3    [washington, reuters, trump, campaign, adviser...
4    [seattle, washington, reuters, president, dona...
Name: cleaned_tokens, dtype: object


In [6]:
# Remove stop words
from nltk.corpus import stopwords # type: ignore
stop_words = set(stopwords.words('english'))

def rem_stop_words(tokens): return [w for w in tokens if w not in stop_words]

fake['cleaned_tokens'] = fake['cleaned_tokens'].apply(rem_stop_words)
real['cleaned_tokens'] = real['cleaned_tokens'].apply(rem_stop_words)

print(f'fake tokenized:\n{fake['cleaned_tokens'].head(5)}')
print(f'\nreal tokenized:\n{real['cleaned_tokens'].head(5)}')



fake tokenized:
0    [donald, trump, wish, americans, happy, new, y...
1    [house, intelligence, committee, chairman, dev...
2    [friday, revealed, former, milwaukee, sheriff,...
3    [christmas, day, donald, trump, announced, wou...
4    [pope, francis, used, annual, christmas, day, ...
Name: cleaned_tokens, dtype: object

real tokenized:
0    [washington, reuters, head, conservative, repu...
1    [washington, reuters, transgender, people, all...
2    [washington, reuters, special, counsel, invest...
3    [washington, reuters, trump, campaign, adviser...
4    [seattle, washington, reuters, president, dona...
Name: cleaned_tokens, dtype: object


In [7]:
# vocabulary vector = unique word tokenize from fake + real
all_words = pd.concat([fake['cleaned_tokens'], real['cleaned_tokens']])
vocabulary = sorted(list(set(word for tokens in all_words for word in tokens)))

print(f"Total unique words in vocabulary: {len(vocabulary)}")



Total unique words in vocabulary: 5810


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# 1. Join tokens back into strings for the Vectorizer
fake_strings = fake['cleaned_tokens'].apply(lambda x: ' '.join(x))
real_strings = real['cleaned_tokens'].apply(lambda x: ' '.join(x))
all_texts = pd.concat([fake_strings, real_strings])

# 2. Initialize and apply CountVectorizer
vectorizer = CountVectorizer(max_features=6000) 
X = vectorizer.fit_transform(all_texts)

y = np.array([0]*len(fake) + [1]*len(real))

print(X.shape)
print(y.shape)


(100, 5785)
(100,)


In [9]:
from sklearn.model_selection import train_test_split
from random import randint
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

# Split X y
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=randint(0, 1000), stratify=y)

print(f'BoW matrix shape (train): {X_train.shape}')

# Initialize & run model
model = MultinomialNB()
model.fit(X_train, y_train)

# Test + accuracy score
y_pred = model.predict(X_test)

print(f'\nAccuracy: {accuracy_score(y_test, y_pred)}')



BoW matrix shape (train): (80, 5785)

Accuracy: 1.0
